<div class="alert alert-info" markdown="1">

#### Lab 6

# Matrix-Vector Multiplication

### Math 124: Vectors, Matrices, and Applications, Fall 2026

<small><a style="text-decoration: none" href="">math124.org</a></small>
    
</div>


### Instructions

Most homeworks and some labs will have a Jupyter Notebook, like this one, containing Python code that supplements our understanding of the relevant mathematical ideas of the week.

---

Open this notebook in Google Colab. Instructions on how to use Google Colab are at [math124.org/running-code](https://math124.org/running-code).

**Most importantly, make sure to click "Copy to Drive" and rename the notebook before proceeding!**

---

Work from top to bottom. In each task, replace `...` with your code. You will only need to write one or two assignments at a time.

To receive credit for the programming component of the lab, complete the tasks and show your lab TA your results and plots.

## Defining a matrix

---

First, run the cell below to import NumPy.

In [ ]:
import numpy as np

We have used `np.array` to define vectors. To define a **matrix**, put each row in its own pair of square brackets, separated by commas, and put another pair of brackets around all the rows.

For example, here is

$$A=\begin{bmatrix}2&-1&3\\4&0&-2\end{bmatrix}.$$

In [ ]:
A = np.array([[2, -1, 3],
              [4,  0, -2]])
A

`A.shape` tells us the number of rows and columns, in that order. This matrix has shape $2 \times 3$.

In [ ]:
A.shape

## Multiplying a matrix by a vector

---

Use `@` to multiply a matrix by a vector. For example, let

$$\vec x=\begin{bmatrix}1\\-2\\3\end{bmatrix}.$$

The matrix $A$ has three columns, so we can multiply it by this vector with three entries. Run the cell below to find $A\vec x$.

In [ ]:
x = np.array([1, -2, 3])
A @ x

The result represents $\begin{bmatrix}13\\-2\end{bmatrix}$.

Use **`@`**, rather than `*`, for matrix-vector multiplication.

<div class="alert alert-warning" markdown="1">

### Task 1: Verify two computations

</div>


Define the following matrix and vector in the next cell:

$$C=\begin{bmatrix}4&-3&6\\-2&5&1\end{bmatrix},\qquad
\vec v=\begin{bmatrix}-2\\3\\4\end{bmatrix}.$$

Then run `C @ v` to verify that $C\vec v=\begin{bmatrix}7\\23\end{bmatrix}$.

In [ ]:
C = ...
v = ...

In [ ]:
C @ v

Now define

$$B=\begin{bmatrix}2&-1\\-3&4\\5&2\end{bmatrix}.$$

Use `@` and parentheses to verify that $B(C\vec v)=\begin{bmatrix}-9\\71\\81\end{bmatrix}$.

In [ ]:
B = ...
B @ (C @ v)

## See what a matrix does

---

Let's look at what a $2\times 2$ matrix does to two vectors in $\mathbb{R}^2$:

$$\vec u=\begin{bmatrix}2\\1\end{bmatrix},\qquad
\vec v=\begin{bmatrix}-1\\2\end{bmatrix}.$$

Run the collapsed cell below once to define **`show_transformation`**. You do not need to edit it. Give this function a matrix `A`, and it will draw $\vec u$ and $\vec v$ on the left and $A\vec u$ and $A\vec v$ on the right.

The function always uses these same two vectors. Orange represents $\vec u$ and its image; blue represents $\vec v$ and its image.

In [ ]:
#@title Load the plotting tools { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("retina")
from IPython.display import display, HTML
import json
import uuid
import base64
from io import BytesIO
from matplotlib.mathtext import math_to_image
from matplotlib.font_manager import FontProperties

_vectors = np.array([[2., 1.], [-1., 2.]])
_colors = ['#e69f00', '#3d81f6']

def show_transformation(A, matrix_name="A"):
    """Show vectors before and after A, labeling outputs with matrix_name."""
    if A is Ellipsis:
        return
    A = np.asarray(A, dtype=float)
    if A.shape != (2, 2):
        raise ValueError('Enter a matrix with shape (2, 2).')
    transformed = np.array([A @ vector for vector in _vectors])
    limit = max(3., 1.35 * np.max(np.abs(np.r_[_vectors, transformed])))
    with plt.rc_context({'font.family': 'serif', 'font.serif': ['Palatino', 'DejaVu Serif'],
                         'font.size': 13, 'axes.titlesize': 15, 'mathtext.fontset': 'cm'}):
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), dpi=180)
        for panel, (ax, vectors) in enumerate(zip(axes, [_vectors, transformed])):
            ax.set(xlim=(-limit, limit), ylim=(-limit, limit),
                   xlabel='First coordinate', ylabel='Second coordinate',
                   title=['Before', 'After'][panel])
            ax.set_aspect('equal')
            ax.grid(color='#e7e7e7', linewidth=.7)
            ax.axhline(0, color='#999', linewidth=1)
            ax.axvline(0, color='#999', linewidth=1)
            for spine in ax.spines.values():
                spine.set_visible(False)
            for i, vector in enumerate(vectors):
                name = [r'\vec u', r'\vec v'][i]
                if panel:
                    name = matrix_name + name
                ax.annotate('', xy=vector, xytext=(0, 0),
                            arrowprops=dict(arrowstyle='->', color=_colors[i], lw=2.4))
                ax.annotate('$' + name + '$', xy=vector, xytext=(7, 8),
                            textcoords='offset points', color=_colors[i], fontsize=15)
        fig.tight_layout(pad=1.5)
        plt.show()

def _matrix_explorer(kind):
    angles = list(range(-180, 181)) if kind == 'rotation' else list(range(181))
    initial = 60 if kind == 'rotation' else 45
    frames = []
    for angle in angles:
        theta = np.deg2rad(angle)
        c, s = np.cos(theta), np.sin(theta)
        M = np.array([[c, -s], [s, c]]) if kind == 'rotation' else np.array([[c*c, c*s], [c*s, s*s]])
        frames.append({'after': [(M @ v).tolist() for v in _vectors], 'line': [c, s]})
    labels = {}
    with plt.rc_context({'mathtext.fontset': 'cm'}):
        for prefix in ['', 'R', 'P']:
            for i, name in enumerate(['u', 'v']):
                buffer = BytesIO()
                math_to_image('$' + prefix + r'\vec ' + name + '$', buffer,
                              prop=FontProperties(size=19), format='svg', color=_colors[i])
                labels[prefix + name] = 'data:image/svg+xml;base64,' + base64.b64encode(buffer.getvalue()).decode()
    data = dict(kind=kind, angles=angles, initial=initial, before=_vectors.tolist(), frames=frames, labels=labels)
    template = r'''<div id="__ID__" style="max-width:1000px;margin:12px auto;color:#202020;font:17px Palatino,Georgia,serif;background:white">
<style>
#__ID__ .matrix {position:relative;display:inline-grid;grid-template-columns:auto auto;gap:8px 22px;padding:5px 12px;vertical-align:middle;text-align:center}
#__ID__ .matrix::before, #__ID__ .matrix::after {content:'';position:absolute;top:0;bottom:0;width:7px;border-top:2px solid;border-bottom:2px solid}
#__ID__ .matrix::before {left:0;border-left:2px solid}
#__ID__ .matrix::after {right:0;border-right:2px solid}
#__ID__ .formula {text-align:center;min-height:95px;padding:12px 0;box-sizing:border-box}
#__ID__ input {width:min(70%,650px);vertical-align:middle;accent-color:#3d81f6}
#__ID__ canvas {display:block;width:100%;height:auto;background:white}
</style>
<label>Angle θ: <input type="range" min="__MIN__" max="180" step="1" value="__INITIAL__" aria-label="Angle theta in degrees"> <strong data-angle></strong></label>
<div class="formula" data-matrix></div>
<canvas width="1000" height="480" role="img" aria-label="Two vectors before and after applying the matrix"></canvas>
</div>
<script>
(() => {
 const root=document.getElementById('__ID__'), data=__DATA__;
 const slider=root.querySelector('input'), canvas=root.querySelector('canvas'), ctx=canvas.getContext('2d');
 const pixelRatio=Math.max(2, window.devicePixelRatio || 1);
 canvas.width=Math.round(1000*pixelRatio);
 canvas.height=Math.round(480*pixelRatio);
 ctx.scale(pixelRatio,pixelRatio);
 const colors=['#e69f00','#3d81f6'], names=['u','v'], limit=3.1;
 const labels={};
 const layouts=[{cx:260,cy:247},{cx:750,cy:247}], scale=57;
 function point(v,p){return [layouts[p].cx+scale*v[0],layouts[p].cy-scale*v[1]];}
 function line(a,b,color,width=1){ctx.beginPath();ctx.moveTo(...a);ctx.lineTo(...b);ctx.strokeStyle=color;ctx.lineWidth=width;ctx.stroke();}
 function label(text,x,y,color='#202020',align='center',font='16px Palatino, Georgia, serif'){
  ctx.font=font;ctx.fillStyle=color;ctx.textAlign=align;ctx.fillText(text,x,y);
 }
 function axes(p,title,direction){
  const {cx,cy}=layouts[p],edge=limit*scale;
  label(title,cx,30,'#202020','center','19px Palatino, Georgia, serif');
  for(let k=-3;k<=3;k++){
   line(point([k,-limit],p),point([k,limit],p),'#e7e7e7');
   line(point([-limit,k],p),point([limit,k],p),'#e7e7e7');
   label(String(k),cx+k*scale,cy+edge+23);
   label(String(k),cx-edge-14,cy-k*scale+5,'#202020','right');
  }
  line(point([-limit,0],p),point([limit,0],p),'#999',1.2);
  line(point([0,-limit],p),point([0,limit],p),'#999',1.2);
  label('First coordinate',cx,471);
  ctx.save();ctx.translate(cx-edge-48,cy);ctx.rotate(-Math.PI/2);label('Second coordinate',0,0);ctx.restore();
  if(direction){
   const extent=limit/Math.max(Math.abs(direction[0]),Math.abs(direction[1]));
   line(point(direction.map(x=>-extent*x),p),point(direction.map(x=>extent*x),p),'#888',1.6);
   const end=point(direction.map(x=>.9*extent*x),p);label('ℓ',end[0]+12,end[1]-8,'#666');
  }
 }
 function arrow(v,p,i,prefix){
  const start=point([0,0],p),end=point(v,p),color=colors[i];
  if(Math.hypot(...v)>1e-8){
   line(start,end,color,3);
   const angle=Math.atan2(end[1]-start[1],end[0]-start[0]);
   ctx.beginPath();ctx.moveTo(...end);
   ctx.lineTo(end[0]-11*Math.cos(angle-.4),end[1]-11*Math.sin(angle-.4));
   ctx.lineTo(end[0]-11*Math.cos(angle+.4),end[1]-11*Math.sin(angle+.4));
   ctx.closePath();ctx.fillStyle=color;ctx.fill();
  }
  const img=labels[prefix+names[i]],x=end[0]+(i?-13:13),y=end[1]+(i?10:-35);
  if(img && img.complete && img.naturalWidth){
   const height=27,width=height*img.naturalWidth/img.naturalHeight;
   ctx.drawImage(img,i?x-width:x,y,width,height);
  }
 }
 function render(){
  const angle=Number(slider.value),f=data.frames[angle-data.angles[0]],d=angle+'°';
  root.querySelector('[data-angle]').textContent=d;
  const c='cos('+d+')',s='sin('+d+')';
  const entries=data.kind==='rotation'?[c,'−'+s,s,c]:['cos²('+d+')',c+s,c+s,'sin²('+d+')'];
  const symbol=data.kind==='rotation'?'R':'P';
  root.querySelector('[data-matrix]').innerHTML=symbol+' = <span class="matrix">'+entries.map(x=>'<span>'+x+'</span>').join('')+'</span>';
  ctx.clearRect(0,0,canvas.width,canvas.height);
  const direction=data.kind==='projection'?f.line:null;
  axes(0,'Before',direction);axes(1,'After',direction);
  data.before.forEach((v,i)=>arrow(v,0,i,''));f.after.forEach((v,i)=>arrow(v,1,i,symbol));
 }
 let pending=false;
 for(const [name,src] of Object.entries(data.labels)){
  const img=new Image();labels[name]=img;img.onload=render;img.src=src;
 }
 slider.addEventListener('input',()=>{if(!pending){pending=true;requestAnimationFrame(()=>{pending=false;render();});}});
 render();
})();
</script>'''
    ident = 'matrix-explorer-' + uuid.uuid4().hex
    html = template.replace('__ID__', ident).replace('__DATA__', json.dumps(data))
    html = html.replace('__MIN__', str(angles[0])).replace('__INITIAL__', str(initial))
    display(HTML(html))

def rotation_explorer():
    """Rotate the fixed vectors with a slider for theta."""
    _matrix_explorer('rotation')

def projection_explorer():
    """Project the fixed vectors onto a line controlled by theta."""
    _matrix_explorer('projection')


For example, this matrix, called the **identity matrix**, leaves both vectors unchanged:

$$I=\begin{bmatrix}1&0\\0&1\end{bmatrix}.$$

In [ ]:
I = np.array([[1, 0], [0, 1]])
show_transformation(I, matrix_name="I")

Here is another example. This matrix changes the sign of the second coordinate, reflecting each vector across the horizontal axis:

$$F=\begin{bmatrix}1&0\\0&-1\end{bmatrix}.$$

In [ ]:
F = np.array([[1, 0], [0, -1]])
show_transformation(F, matrix_name="F")

<div class="alert alert-warning" markdown="1">

### Task 2: Try three matrices

</div>


For each matrix below, replace `...` with its NumPy definition and run the cell. Before running it, predict where each vector will end up. Describe what changes in the **After** plot.

**(a)** $\displaystyle D=\begin{bmatrix}2&0\\0&\frac12\end{bmatrix}$

In [ ]:
D = ...
show_transformation(D, matrix_name="D")

**(b)** $\displaystyle S=\begin{bmatrix}0&1\\1&0\end{bmatrix}$

In [ ]:
S = ...
show_transformation(S, matrix_name="S")

**(c)** $\displaystyle H=\begin{bmatrix}1&1\\0&1\end{bmatrix}$

In [ ]:
H = ...
show_transformation(H, matrix_name="H")

## A closer look: rotations

---

The matrix

$$R=\begin{bmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{bmatrix}$$

rotates vectors in $\mathbb{R}^2$ by an angle $\theta$. Positive angles give counterclockwise rotations; negative angles give clockwise rotations.

Run the next cell and move the slider. The **Before** plot keeps the same $\vec u$ and $\vec v$ as above; the **After** plot shows $R\vec u$ and $R\vec v$. The matrix above the plots updates with the angle, starting at $60^\circ$.

In [ ]:
rotation_explorer()

<div class="alert alert-warning" markdown="1">

### Task 3: Explore rotations

</div>


Try $0^\circ$, $90^\circ$, and $-90^\circ$. For each angle, compare the two plots.

What happens to the lengths of the vectors? What happens to the angle between them? Discuss with your group.

## A closer look: projections

---

A matrix can also project vectors onto a line. If the unit vector $\vec w=\begin{bmatrix}w_1\\w_2\end{bmatrix}$ spans a line $\ell$, the matrix

$$P=\begin{bmatrix}w_1^2&w_1w_2\\w_1w_2&w_2^2\end{bmatrix}$$

projects vectors onto $\ell$. Multiplying $P$ by a vector gives that vector's projection onto the line.

<div class="alert alert-warning" markdown="1">

### Task 4: Enter a projection matrix

</div>


Let $\ell$ be the line spanned by $\vec w=\begin{bmatrix}3/5\\4/5\end{bmatrix}$, which has length $1$.

Use the formula above to define `P`, then run the cell to see what it does to the same fixed $\vec u$ and $\vec v$. You can type fractions directly in Python, such as `3/5`.

Where do both vectors in the **After** plot lie?

In [ ]:
P = ...
show_transformation(P, matrix_name="P")

Now let's change the line. A unit vector making an angle $\theta$ with the horizontal axis is

$$\vec w=\begin{bmatrix}\cos\theta\\\sin\theta\end{bmatrix}.$$

Run the cell below. Move the slider to turn $\ell$ and update its projection matrix $P$. The gray line is $\ell$; the **After** plot shows $P\vec u$ and $P\vec v$.

In [ ]:
projection_explorer()

**Your turn:** Try a horizontal line, a vertical line, and a tilted line. What happens when one of the original vectors already lies on $\ell$? What happens when it is perpendicular to $\ell$?

---

When you are finished, show your lab TA your completed tasks and plots.